<a href="https://colab.research.google.com/github/Ghana-Data-Science-Summit-IndabaX-Ghana/IndabaX-2026/blob/main/Ethical%20Computer%20Vision/notebook/Part2.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# The Audit
### Building Ethical Vision Systems — Ghana Data Science Summit 2026
*Part 2 · Led by Emmanuel Asante*

---

A health startup in Accra has built a skin-condition screening app for community health
workers in rural Ghana. The plan is to fine-tune a small, phone-friendly model, check the
benchmark numbers, and ship.

**Your job:** train the model, audit it properly, document where it fails, and make a
recommendation. Who was in the training data? Who wasn't? And what happens when the model
meets a patient it was never trained to recognise?

**What you'll build:**
- You'll fine-tune **MobileNetV3-Small** — a backbone small enough to run on a community
  health worker's phone — on HAM10000, a real dermatology dataset.
- You'll then stress-test it against real dermoscopic images of **confirmed-benign lesions on
  dark skin tones** (Fitzpatrick IV–VI), from a Memorial Sloan Kettering dataset built
  specifically for skin-tone fairness research.
- That stress-test set has **no malignant/benign diagnosis labels** — every lesion in it was
  already cleared by a dermatologist. That constraint becomes the audit's sharpest tool: not
  "is the model accurate on dark skin," but **"does the model start crying wolf on dark
  skin?"** A model that suddenly sees melanoma everywhere on Fitzpatrick V–VI, when every one
  of those lesions was confirmed benign, is a model nobody can trust.

**How to use this notebook:**
1. Run the setup cells in Section 00 — they install dependencies and pull both datasets and a pre-trained checkpoint from Hugging Face
2. Work through each section top to bottom
3. Stop at every `✍️ Reflection` block and write your answers in the markdown cell below it
4. Your final deliverable is a recommendation at the end of Section 06

---
## ⚙️ SECTION 00 — SETUP
*Run these cells once — they take about 3-5 minutes total, no accounts or API keys required.*

---

In [ ]:
# Install dependencies — takes 1-2 minutes on a fresh Colab instance
%pip install -q torch torchvision timm numpy pandas matplotlib seaborn \
             scikit-learn pillow tqdm huggingface_hub
print("✓ Dependencies ready")

In [ ]:
# GPU check — a GPU isn't required for this notebook (we load a pre-trained checkpoint),
# but the optional live training demo in Section 02 runs faster with one.
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — fine for this notebook, the 2-epoch demo just runs a bit slower.")

In [ ]:
# Clone the repo (for the HAM10000 datasheet and config files)
import os
if not os.path.exists("IndabaX-2026"):
    !git clone https://github.com/Ghana-Data-Science-Summit-IndabaX-Ghana/IndabaX-2026.git IndabaX-2026 --depth 1 -q
    print("✓ Repo cloned")
else:
    print("✓ Repo already present")

### Pulling the datasets and checkpoint from Hugging Face

Both datasets — HAM10000 and the ISIC 413 dark-skin-tone stress-test set — and the
pre-trained MobileNetV3-Small checkpoint are hosted on Hugging Face for this session, so
nobody needs a Kaggle account or an ISIC Archive login. This is the single biggest time-saver
in the setup: no auth tokens, no account creation, no waiting on a third-party API mid-session.

If any of these repos are private, you may be prompted for a Hugging Face token — get a free
one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (read access
is enough) and run `huggingface-cli login` or paste it into the prompt.

Both datasets are stored as a single zip archive each (rather than thousands of loose image
files) — this matters in practice: downloading one large file is dramatically faster than
Hugging Face's hub having to check and fetch thousands of individual files one at a time.

In [ ]:
# Download HAM10000 from Hugging Face — single zip archive, not loose files
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_DIR = Path("data")
HAM_DIR = DATA_DIR / "ham10000"
HAM_REPO = "yungprof/ham10000-indabax"

if not HAM_DIR.exists() or not any(HAM_DIR.glob("**/*.csv")):
    zip_path = hf_hub_download(repo_id=HAM_REPO, repo_type="dataset", filename="ham10000.zip")
    HAM_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(HAM_DIR)
    print("✓ HAM10000 downloaded and extracted")
else:
    print("✓ HAM10000 already present")

print(f"HAM10000 ready at: {HAM_DIR}")
!find {str(HAM_DIR)} -maxdepth 2 -type d

### The dark-skin-tone stress-test set (ISIC Collection 413)

This is the **MSKCC Skin Tone Labeling Dataset** — 4,879 dermoscopic images from 64 patients,
recruited specifically to span all six Fitzpatrick skin types. Every lesion in it was
confirmed benign by a board-certified dermatologist at the time of imaging — there is no
melanoma in this set. That matters for how we use it later: **we are not testing diagnostic
accuracy here, we are testing whether the model stays calm on skin tones it never trained on.**

In [ ]:
# Download ISIC Collection 413 from Hugging Face — single zip archive, not loose files
ISIC_DIR = DATA_DIR / "isic_413_skintone"
ISIC_REPO = "yungprof/isic413-skintone-indabax"

if not ISIC_DIR.exists() or not any(ISIC_DIR.glob("*.csv")):
    zip_path = hf_hub_download(repo_id=ISIC_REPO, repo_type="dataset", filename="isic413.zip")
    ISIC_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ISIC_DIR)
    print("✓ ISIC Collection 413 downloaded and extracted")
else:
    print("✓ ISIC Collection 413 already present")

print(f"ISIC Collection 413 ready at: {ISIC_DIR}")
!find {str(ISIC_DIR)} -maxdepth 2 | head -20

In [ ]:
# Core imports and shared helpers — run once
import warnings; warnings.filterwarnings("ignore")
import os, json, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                              precision_recall_curve, roc_auc_score)

# Palette
TEAL  = "#028090"
NAVY  = "#1A2535"
AMBER = "#D97706"
RED   = "#DC2626"
GREY  = "#9CA3AF"

# Class definitions — HAM10000's 7 diagnostic classes
CLASS_NAMES  = ["nv", "mel", "bcc", "akiec", "bkl", "df", "vasc"]
CLASS_LABELS = {
    "nv":    "Melanocytic Nevi",
    "mel":   "Melanoma",
    "bcc":   "Basal Cell Carcinoma",
    "akiec": "Actinic Keratoses",
    "bkl":   "Benign Keratoses",
    "df":    "Dermatofibroma",
    "vasc":  "Vascular Lesions",
}
MALIGNANT = {"mel", "bcc", "akiec"}

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Core helpers loaded | device = {DEVICE}")

## Building the training pipeline

We're fine-tuning **MobileNetV3-Small**, pretrained on ImageNet, with its classifier head
replaced for 7-way lesion classification. MobileNetV3-Small is the right backbone for this
use case for a concrete reason: it has roughly **2.5 million parameters** versus EfficientNet-B0's
~5.3 million, and it was designed from the ground up for on-device, low-latency inference on
phones — which is exactly the deployment target a community health worker app would need.
A model that only runs well on a GPU server isn't actually deployable in a rural clinic with
patchy connectivity.

In [ ]:
# Transforms — light augmentation for training, none for eval
TRAIN_TRANSFORM = T.Compose([
    T.RandomResizedCrop(224, scale=(0.85, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

EVAL_TRANSFORM = T.Compose([
    T.Resize(224),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class HAM10000Dataset(Dataset):
    """HAM10000 train/val/test split. Expects a dataframe with image_id + dx columns
    and a dict mapping image_id -> filepath (HAM10000 ships images across two folders)."""
    def __init__(self, df, path_lookup, transform):
        self.df = df.reset_index(drop=True)
        self.path_lookup = path_lookup
        self.transform = transform
        self.labels = [CLASS_NAMES.index(dx) for dx in self.df["dx"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        image_id = self.df.iloc[i]["image_id"]
        img = Image.open(self.path_lookup[image_id]).convert("RGB")
        return self.transform(img), self.labels[i], image_id


class StressTestDataset(Dataset):
    """ISIC 413 dark-skin-tone stress test set. No diagnosis labels — every lesion is
    confirmed benign. We still need a Dataset to batch images through the model."""
    def __init__(self, image_paths, fitzpatrick_lookup, transform):
        self.image_paths = image_paths
        self.fitzpatrick_lookup = fitzpatrick_lookup
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, i):
        path = self.image_paths[i]
        image_id = path.stem
        img = Image.open(path).convert("RGB")
        fitz = self.fitzpatrick_lookup.get(image_id, None)
        return self.transform(img), image_id, fitz if fitz is not None else -1


def build_mobilenet_v3_small(num_classes=7, pretrained=True):
    """MobileNetV3-Small with a fresh classifier head for lesion classification."""
    model = tvm.mobilenet_v3_small(weights="IMAGENET1K_V1" if pretrained else None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


print("✓ Dataset classes and model builder defined")

In [ ]:
# ── Inference & metrics helpers ───────────────────────────────────────────────
def run_inference(model, loader, device):
    """Run a full inference pass. Returns predictions/probs/labels/ids."""
    preds, probs_all, labels_all, ids_all = [], [], [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference", unit="batch"):
            imgs, lbls, ids = batch
            p = torch.softmax(model(imgs.to(device)), 1)
            preds.extend(p.argmax(1).cpu().tolist())
            probs_all.extend(p.cpu().tolist())
            labels_all.extend(lbls.tolist() if torch.is_tensor(lbls) else list(lbls))
            ids_all.extend(list(ids))
    return {"predictions": preds, "probabilities": probs_all,
            "true_labels": labels_all, "image_ids": ids_all}


def run_inference_unlabeled(model, loader, device):
    """Inference pass for the stress-test set, which has no ground-truth diagnosis."""
    preds, probs_all, ids_all, fitz_all = [], [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, ids, fitz in tqdm(loader, desc="Stress-test inference", unit="batch"):
            p = torch.softmax(model(imgs.to(device)), 1)
            preds.extend(p.argmax(1).cpu().tolist())
            probs_all.extend(p.cpu().tolist())
            ids_all.extend(list(ids))
            fitz_all.extend(fitz.tolist() if torch.is_tensor(fitz) else list(fitz))
    return {"predictions": preds, "probabilities": probs_all,
            "image_ids": ids_all, "fitzpatrick": fitz_all}


def compute_metrics(preds, true_labels, probs):
    p, y = np.array(preds), np.array(true_labels)
    pr = np.array(probs)
    per_cls_acc = {}
    for i, cls in enumerate(CLASS_NAMES):
        m = y == i
        per_cls_acc[cls] = float(accuracy_score(y[m], p[m])) if m.sum() > 0 else float("nan")
    pcf = f1_score(y, p, average=None, zero_division=0, labels=list(range(7)))
    max_p = pr.max(1)
    correct = p == y
    return {
        "overall_accuracy": float(accuracy_score(y, p)),
        "macro_f1": float(f1_score(y, p, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y, p, average="weighted", zero_division=0)),
        "per_class_accuracy": per_cls_acc,
        "per_class_f1": {cls: float(pcf[i]) for i, cls in enumerate(CLASS_NAMES)},
        "confusion_matrix": confusion_matrix(y, p, labels=list(range(7))),
        "avg_confidence_correct": float(max_p[correct].mean()) if correct.any() else float("nan"),
        "avg_confidence_wrong": float(max_p[~correct].mean()) if (~correct).any() else float("nan"),
    }


def build_failure_table(preds, true_labels, probs, image_ids, conf_thresh=0.80):
    rows = []
    for img_id, pred, true, prob in zip(image_ids, preds, true_labels, probs):
        conf = float(max(prob))
        pred_cls, true_cls = CLASS_NAMES[pred], CLASS_NAMES[true]
        if pred == true:
            etype = "correct"
        elif conf >= conf_thresh:
            etype = "high_conf_wrong"
        elif true_cls in MALIGNANT and pred_cls not in MALIGNANT:
            etype = "false_negative"
        elif true_cls not in MALIGNANT and pred_cls in MALIGNANT:
            etype = "false_positive"
        else:
            etype = "wrong_other"
        rows.append({"image_id": img_id, "true_label": CLASS_LABELS[true_cls],
                     "predicted_label": CLASS_LABELS[pred_cls],
                     "confidence": round(conf, 4), "error_type": etype})
    return pd.DataFrame(rows).sort_values("confidence", ascending=False).reset_index(drop=True)


print("✓ Inference and metrics helpers defined")

In [ ]:
# ── Plotting helpers ──────────────────────────────────────────────────────────
def _style():
    plt.style.use("seaborn-v0_8-whitegrid")

def plot_confusion_matrix(true_labels, preds, title="Confusion Matrix", figsize=(9,7)):
    _style()
    labels = [CLASS_LABELS[c] for c in CLASS_NAMES]
    cm = confusion_matrix(true_labels, preds, labels=list(range(7)))
    cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(1, keepdims=True))
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap=sns.light_palette(TEAL, as_cmap=True),
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.5, linecolor="#F3F4F6", cbar_kws={"shrink":0.8})
    ax.set_title(title, fontsize=14, fontweight="bold", color=NAVY, pad=12)
    ax.set_ylabel("True Label", fontsize=11, color=NAVY)
    ax.set_xlabel("Predicted Label", fontsize=11, color=NAVY)
    ax.tick_params(labelsize=8)
    plt.tight_layout(); plt.show()

def plot_per_class_f1(metrics, title="Per-Class F1 Score", figsize=(9,4)):
    _style()
    items = sorted(metrics["per_class_f1"].items(), key=lambda x: x[1], reverse=True)
    classes, scores = zip(*items)
    names = [CLASS_LABELS[c] for c in classes]
    colours = [AMBER if s < 0.5 else TEAL for s in scores]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(names, scores, color=colours, edgecolor="white")
    for bar, s in zip(bars, scores):
        ax.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                f"{s:.2f}", va="center", ha="left", fontsize=9, color=NAVY)
    ax.set_xlim(0, 1.15); ax.set_xlabel("F1 Score", color=NAVY)
    ax.set_title(title, fontsize=13, fontweight="bold", color=NAVY, pad=10)
    ax.axvline(0.5, color=AMBER, ls="--", lw=1, alpha=0.7, label="F1 = 0.50")
    ax.legend(fontsize=8); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

def plot_training_curves(history, figsize=(12,4)):
    _style()
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    epochs = range(1, len(history["train_loss"])+1)
    axes[0].plot(epochs, history["train_loss"], color=TEAL, marker="o", label="Train")
    axes[0].plot(epochs, history["val_loss"], color=AMBER, marker="o", label="Validation")
    axes[0].set_title("Loss", fontweight="bold", color=NAVY); axes[0].set_xlabel("Epoch")
    axes[0].legend(fontsize=9)
    axes[1].plot(epochs, history["train_acc"], color=TEAL, marker="o", label="Train")
    axes[1].plot(epochs, history["val_acc"], color=AMBER, marker="o", label="Validation")
    axes[1].set_title("Accuracy", fontweight="bold", color=NAVY); axes[1].set_xlabel("Epoch")
    axes[1].legend(fontsize=9)
    plt.tight_layout(); plt.show()

def plot_precision_recall_with_threshold(probs, true_labels, chosen_thresh=0.80, figsize=(8,5)):
    """Binarised malignant-vs-benign precision/recall, with the chosen confidence
    threshold marked, so the threshold can be defended rather than asserted."""
    _style()
    y = np.array(true_labels)
    pr = np.array(probs)
    malignant_idx = [CLASS_NAMES.index(c) for c in MALIGNANT]
    y_bin = np.isin(y, malignant_idx).astype(int)
    malignant_score = pr[:, malignant_idx].sum(axis=1)

    precision, recall, thresholds = precision_recall_curve(y_bin, malignant_score)
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(thresholds, precision[:-1], color=TEAL, label="Precision", lw=2)
    ax.plot(thresholds, recall[:-1], color=AMBER, label="Recall", lw=2)
    ax.axvline(chosen_thresh, color=RED, ls="--", lw=1.5,
               label=f"Chosen threshold = {chosen_thresh:.2f}")
    idx = np.argmin(np.abs(thresholds - chosen_thresh))
    ax.scatter([thresholds[idx]], [precision[idx]], color=TEAL, zorder=5, s=60)
    ax.scatter([thresholds[idx]], [recall[idx]], color=AMBER, zorder=5, s=60)
    ax.annotate(f"P={precision[idx]:.2f}", (thresholds[idx], precision[idx]),
                textcoords="offset points", xytext=(8,8), fontsize=9, color=TEAL)
    ax.annotate(f"R={recall[idx]:.2f}", (thresholds[idx], recall[idx]),
                textcoords="offset points", xytext=(8,-14), fontsize=9, color=AMBER)
    ax.set_xlabel("Malignant-probability threshold", color=NAVY)
    ax.set_ylabel("Score", color=NAVY)
    ax.set_title("Precision / Recall vs Threshold — Malignant Detection",
                 fontsize=13, fontweight="bold", color=NAVY, pad=10)
    ax.legend(fontsize=9); ax.set_ylim(0, 1.05)
    plt.tight_layout(); plt.show()
    return dict(zip(["thresholds","precision","recall"], [thresholds, precision[:-1], recall[:-1]]))

def plot_fitzpatrick_distribution(distribution, dataset_name="Dataset", figsize=(7,4)):
    _style()
    types = sorted([t for t in distribution if t and t > 0])
    total = sum(distribution[t] for t in types)
    pcts = [distribution[t]/total*100 for t in types]
    cols = [AMBER if t >= 5 else TEAL for t in types]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar([f"Type {t}" for t in types], pcts, color=cols, edgecolor="white")
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{pct:.1f}%", ha="center", va="bottom", fontsize=9, color=NAVY)
    ax.set_ylabel("% of Dataset", color=NAVY)
    ax.set_title(f"Fitzpatrick Skin Type Distribution — {dataset_name}",
                 fontsize=12, fontweight="bold", color=NAVY, pad=10)
    handles = [mpatches.Patch(color=TEAL, label="Types I-IV"),
               mpatches.Patch(color=AMBER, label="Types V-VI (underrepresented in training)")]
    ax.legend(handles=handles, fontsize=8)
    ax.set_ylim(0, max(pcts)*1.3 if pcts else 1)
    plt.tight_layout(); plt.show()

def plot_overdiagnosis_by_fitzpatrick(stress_table, figsize=(9,5)):
    """The central plot for the stress test: every lesion here is confirmed benign,
    so any 'malignant' prediction is by definition a false alarm. We plot the
    false-alarm rate by Fitzpatrick type."""
    _style()
    g = stress_table.groupby("fitzpatrick").agg(
        n=("predicted_malignant", "count"),
        false_alarm_rate=("predicted_malignant", "mean"),
        avg_confidence=("confidence", "mean"),
    ).reset_index()
    g = g[g["fitzpatrick"] > 0].sort_values("fitzpatrick")
    cols = [AMBER if t >= 5 else TEAL for t in g["fitzpatrick"]]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar([f"Type {int(t)}\n(n={int(n)})" for t, n in zip(g["fitzpatrick"], g["n"])],
                  g["false_alarm_rate"]*100, color=cols, edgecolor="white")
    for bar, rate in zip(bars, g["false_alarm_rate"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{rate*100:.1f}%", ha="center", va="bottom", fontsize=9, color=NAVY)
    ax.set_ylabel("False-alarm rate (%)\n(confirmed-benign lesions flagged malignant)", color=NAVY)
    ax.set_title("Over-Diagnosis Rate by Fitzpatrick Skin Type — Stress Test",
                 fontsize=13, fontweight="bold", color=NAVY, pad=10)
    handles = [mpatches.Patch(color=TEAL, label="Types I-IV"),
               mpatches.Patch(color=AMBER, label="Types V-VI")]
    ax.legend(handles=handles, fontsize=9)
    plt.tight_layout(); plt.show()
    return g

def plot_confidence_by_fitzpatrick(stress_table, figsize=(9,5)):
    _style()
    g = stress_table[stress_table["fitzpatrick"] > 0]
    fig, ax = plt.subplots(figsize=figsize)
    types = sorted(g["fitzpatrick"].unique())
    data = [g[g["fitzpatrick"]==t]["confidence"] for t in types]
    bp = ax.boxplot(data, labels=[f"Type {int(t)}" for t in types], patch_artist=True)
    for patch, t in zip(bp["boxes"], types):
        patch.set_facecolor(AMBER if t >= 5 else TEAL)
        patch.set_alpha(0.7)
    ax.axhline(0.80, color=RED, ls="--", lw=1.2, label="Threshold 0.80")
    ax.set_ylabel("Model confidence on top prediction", color=NAVY)
    ax.set_title("Model Confidence by Fitzpatrick Type — Stress Test\n(high confidence here is concerning: every lesion is confirmed benign)",
                 fontsize=12, fontweight="bold", color=NAVY, pad=10)
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

def display_failure_grid(images, pred_names, true_names, confidences,
                          fitzpatrick_types=None, n=8, title="Failures", figsize=(14,7)):
    _style()
    n = min(n, len(images))
    cols, rows = 4, (n+3)//4
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    MEAN = np.array([0.485,0.456,0.406]); STD = np.array([0.229,0.224,0.225])
    for i in range(n):
        ax, img = axes[i], images[i]
        if isinstance(img, torch.Tensor): img = img.numpy()
        if isinstance(img, np.ndarray) and img.ndim == 3 and img.shape[0] == 3:
            img = np.transpose(img, (1,2,0)); img = (img * STD + MEAN).clip(0,1)
        ax.imshow(img); ax.axis("off")
        colour = RED if pred_names[i] != true_names[i] else TEAL
        fitz = f"\nFitzpatrick {fitzpatrick_types[i]}" if fitzpatrick_types and fitzpatrick_types[i] else ""
        ax.set_title(f"True: {true_names[i]}", fontsize=7, color=TEAL, pad=2)
        ax.text(0.5, -0.05, f"Pred: {pred_names[i]} ({confidences[i]:.0%}){fitz}",
                transform=ax.transAxes, fontsize=7, color=colour, ha="center", va="top")
    for j in range(n, len(axes)): axes[j].axis("off")
    fig.suptitle(title, fontsize=13, fontweight="bold", color=NAVY, y=1.01)
    plt.tight_layout(); plt.show()

def display_stress_grid(images, pred_names, confidences, fitzpatrick_types, n=8,
                         title="Stress Test Predictions", figsize=(14,7)):
    """Like display_failure_grid, but there's no ground truth — every image is
    confirmed benign, so we display what the model predicted and how confidently."""
    _style()
    n = min(n, len(images))
    cols, rows = 4, (n+3)//4
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    MEAN = np.array([0.485,0.456,0.406]); STD = np.array([0.229,0.224,0.225])
    for i in range(n):
        ax, img = axes[i], images[i]
        if isinstance(img, torch.Tensor): img = img.numpy()
        if isinstance(img, np.ndarray) and img.ndim == 3 and img.shape[0] == 3:
            img = np.transpose(img, (1,2,0)); img = (img * STD + MEAN).clip(0,1)
        ax.imshow(img); ax.axis("off")
        is_alarm = pred_names[i] in [CLASS_LABELS[c] for c in MALIGNANT]
        colour = RED if is_alarm else TEAL
        ax.set_title(f"Fitzpatrick {fitzpatrick_types[i]} · confirmed benign", fontsize=7, color=NAVY, pad=2)
        ax.text(0.5, -0.05, f"Model says: {pred_names[i]} ({confidences[i]:.0%})",
                transform=ax.transAxes, fontsize=7, color=colour, ha="center", va="top")
    for j in range(n, len(axes)): axes[j].axis("off")
    fig.suptitle(title, fontsize=13, fontweight="bold", color=NAVY, y=1.01)
    plt.tight_layout(); plt.show()

print("✓ All plotting helpers loaded")

---
## 📁 SECTION 01 — LOAD AND SPLIT THE TRAINING DATA

---

HAM10000 contains 10,015 dermatoscopic images across 7 diagnostic classes, collected
primarily from clinics in Austria and Australia. We'll split it into train/validation/test
the standard way — stratified by class, so rare classes like dermatofibroma don't disappear
from one split entirely.

In [ ]:
# Locate metadata and build an image_id -> path lookup (HAM10000 ships images split
# across two folders on Kaggle: HAM10000_images_part_1 and _part_2)
HAM_METADATA_PATH = next(HAM_DIR.glob("**/*metadata*.csv"))
ham_df = pd.read_csv(HAM_METADATA_PATH)
print(f"✓ Loaded metadata: {len(ham_df)} rows")

image_folders = [p for p in HAM_DIR.glob("**/*") if p.is_dir() and "image" in p.name.lower()]
path_lookup = {}
for folder in image_folders:
    for img_path in folder.glob("*.jpg"):
        path_lookup[img_path.stem] = img_path

ham_df = ham_df[ham_df["image_id"].isin(path_lookup.keys())].reset_index(drop=True)
print(f"✓ Matched {len(ham_df)} images to files on disk")
print("\nClass distribution:")
print(ham_df["dx"].value_counts())

In [ ]:
# Stratified train / val / test split (70 / 15 / 15)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(ham_df, test_size=0.30, stratify=ham_df["dx"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["dx"], random_state=SEED)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

train_ds = HAM10000Dataset(train_df, path_lookup, TRAIN_TRANSFORM)
val_ds   = HAM10000Dataset(val_df,   path_lookup, EVAL_TRANSFORM)
test_ds  = HAM10000Dataset(test_df,  path_lookup, EVAL_TRANSFORM)

BATCH_SIZE = 32
NUM_WORKERS = 2  # set to 0 if you get DataLoader errors on Windows/Mac
CONFIDENCE_THRESH = 0.80  # revisited and justified later via the precision/recall curve

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("✓ DataLoaders ready")

---
#### 🧮 How metrics fail an auditor
*Two or three minutes of reading before we touch a single number.*

---

You're about to compute accuracy, macro F1, weighted F1, and apply a confidence threshold.
Each one is useful — and each one breaks under a specific, predictable condition. Knowing the
condition is what separates reading a metric from auditing one.

- **Accuracy fails when classes are...** imbalanced. HAM10000 is roughly two-thirds
  melanocytic nevi. A model that always predicts "nevi" scores high accuracy while never once
  catching a melanoma.

- **Weighted F1 fails for what reason?** It weights each class's F1 by how many examples it
  has — which means it rewards the model for being good at the *common* classes and quietly
  buries how it does on the *rare, dangerous* ones. A model can post a strong weighted F1 while
  missing melanoma almost entirely.

- **Confidence fails under distribution shift; under what condition?** When the test
  images stop resembling the training images — different camera, different lighting,
  different skin tone — the model is still forced to output a probability distribution that
  sums to 1. It has no mechanism for saying "I don't know." High confidence on
  out-of-distribution input is not a sign of competence; it's a sign the model wasn't built to
  recognise its own blind spots.

- **A single threshold fails when the model is miscalibrated; why?** A threshold like 0.80
  assumes that "0.80 confident" means roughly the same thing everywhere. If the model is
  systematically over-confident on certain inputs (we'll test exactly this on dark skin tones
  later), the same threshold lets through a different number of dangerous errors depending on
  who the patient is.

*Want to go deeper? Google's [PAIR guide on ML fairness](https://pair.withgoogle.com/) and the
[scikit-learn calibration documentation](https://scikit-learn.org/stable/modules/calibration.html)
are good places to start on your own time.*

---

---
## 🏋️ SECTION 02 — TRAIN MOBILENETV3-SMALL
*This is the part of the audit nobody skips responsibly: knowing exactly what the model saw.*

---

We fine-tune from ImageNet-pretrained weights rather than training from scratch — this is
standard practice for medical imaging with a dataset this size (10,015 images is small by deep
learning standards), and it's also the realistic path a startup would actually take.

**What happens in this section:** the cell below runs **2 live epochs** so you can watch
training actually happen — the loss dropping, accuracy climbing, a real optimizer doing real
work. Two epochs takes a few minutes and is enough to see the mechanics, but it is **not**
enough to produce a properly converged model. Right after, we load a **fully-trained
checkpoint** (8 epochs, same architecture, same data) from Hugging Face — that's the model
every section from here on actually audits. A 2-epoch model would give you a much rosier
picture of bias than an 8-epoch one, so it would be dishonest to audit it instead of the real
thing.

In [ ]:
# Build the model, optimizer, and loss function
DEMO_EPOCHS = 2  # live demo only — see note above
LEARNING_RATE = 3e-4

model = build_mobilenet_v3_small(num_classes=7, pretrained=True).to(DEVICE)

# Class weights — HAM10000 is heavily imbalanced toward melanocytic nevi, so we weight
# the loss to stop the model from just learning to predict the majority class
class_counts = train_df["dx"].value_counts()
weights = torch.tensor([1.0 / class_counts[c] for c in CLASS_NAMES], dtype=torch.float32)
weights = (weights / weights.sum() * len(CLASS_NAMES)).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=DEMO_EPOCHS)

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ MobileNetV3-Small built — {n_params:,} parameters")
print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.cpu().numpy().round(2)))}")

In [ ]:
# Training loop — runs live for DEMO_EPOCHS epochs so you can watch it happen
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for imgs, lbls, _ in tqdm(loader, desc="train" if is_train else "val", leave=False):
            imgs, lbls = imgs.to(device), torch.as_tensor(lbls).to(device)
            if is_train: optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            total_correct += (out.argmax(1) == lbls).sum().item()
            total_n += imgs.size(0)
    return total_loss / total_n, total_correct / total_n


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, DEMO_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step()
    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc); history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}/{DEMO_EPOCHS} | train_loss {train_loss:.3f} acc {train_acc:.1%} | "
          f"val_loss {val_loss:.3f} acc {val_acc:.1%}")

print(f"\n✓ Live demo complete — {DEMO_EPOCHS} epochs. Loss should already be dropping.")
print("  This model is NOT what we'll audit — see the next cell.")

In [ ]:
# Visualise the demo's training curve, just to see the mechanics in action
plot_training_curves(history)

### Loading the fully-trained checkpoint

The cells above showed training happening. Everything from here on uses a **separately
fine-tuned, fully-converged MobileNetV3-Small checkpoint** (8 epochs on the same HAM10000
split, same architecture, same class weighting) hosted on Hugging Face — so the audit results
you see reflect a properly trained model, not a 2-epoch warm-up.

In [ ]:
# Load the fully-trained checkpoint from Hugging Face
from huggingface_hub import hf_hub_download

MODEL_REPO = "yungprof/mobilenetv3-small-ham10000-indabax"
CHECKPOINT_FILENAME = "mobilenetv3_small_ham10000_best.pth"

checkpoint_path = hf_hub_download(repo_id=MODEL_REPO, filename=CHECKPOINT_FILENAME)
ckpt = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✓ Loaded fully-trained checkpoint (epoch {ckpt['epoch']}, val acc {ckpt['val_acc']:.1%})")
print("  This is the model the rest of the notebook audits.")

---
## 📊 SECTION 03 — THE BASELINE RUN
*Evaluate on HAM10000's own held-out test set first — the model's home turf.*

---

In [ ]:
# Run inference on the HAM10000 held-out test split
ham_results = run_inference(model, test_loader, DEVICE)
ham_metrics = compute_metrics(ham_results["predictions"], ham_results["true_labels"], ham_results["probabilities"])

print("HAM10000 held-out test results:")
print(f"  Overall accuracy : {ham_metrics['overall_accuracy']:.1%}")
print(f"  Macro F1         : {ham_metrics['macro_f1']:.3f}")
print(f"  Weighted F1      : {ham_metrics['weighted_f1']:.3f}")
print(f"  Avg confidence when correct : {ham_metrics['avg_confidence_correct']:.1%}")
print(f"  Avg confidence when wrong   : {ham_metrics['avg_confidence_wrong']:.1%}")

In [ ]:
plot_confusion_matrix(ham_results["true_labels"], ham_results["predictions"],
                      title="Confusion Matrix — HAM10000 Held-Out Test Set")

In [ ]:
plot_per_class_f1(ham_metrics, title="Per-Class F1 — HAM10000 Held-Out Test Set")

---
#### 💻 Coding challenge — compute macro F1 by hand

Before trusting `sklearn`'s `f1_score(average="macro")`, compute it yourself from the
confusion matrix. For each class: precision = TP / (TP + FP), recall = TP / (TP + FN),
F1 = 2 · precision · recall / (precision + recall). Macro F1 is the unweighted mean across
all 7 classes. Fill in the function below — the mismatch (or match) with the library result
teaches you more than the formula alone.

---

In [ ]:
# YOUR CODE — implement macro F1 from the confusion matrix
def macro_f1_by_hand(cm):
    """cm: a 7x7 confusion matrix (rows = true class, cols = predicted class).
    Return the macro-averaged F1 score across all 7 classes."""
    n_classes = cm.shape[0]
    f1_scores = []
    for i in range(n_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)
    return float(np.mean(f1_scores))


your_macro_f1 = macro_f1_by_hand(ham_metrics["confusion_matrix"])
library_macro_f1 = ham_metrics["macro_f1"]

print(f"Your by-hand macro F1   : {your_macro_f1:.4f}")
print(f"sklearn's macro F1      : {library_macro_f1:.4f}")
print(f"Match: {'✓ yes' if abs(your_macro_f1 - library_macro_f1) < 1e-3 else '✗ no — check your formula'}")

---
#### 🗳️ Quick poll — before you see the numbers

*Discuss with the room: based on the per-class F1 chart above, would you deploy this model
as-is for melanoma screening? Yes / No / It depends. Note your answer — you'll revisit it in
Reflection 1.*

---

---
#### ✍️ Reflection 1 — First Impressions

*Write your answers in the cell below. There is no right answer — this is your analysis.*

1. The model reports a headline accuracy figure. Is that number trustworthy as a safety metric? Why or why not?
2. Which class has the lowest F1 score? What does that mean clinically if this model is used for screening?
3. **What overall accuracy threshold would change your decision to deploy?** Write down a number now — you'll compare it against the African-context gap in Reflection 2.

---

*Your answer here.*

---

---
## 📉 Where did the 0.80 confidence threshold come from?
*It's been used implicitly throughout. Let's justify it — or replace it.*

---

A confidence threshold of 0.80 has been doing real work: it's what separates a "routine"
prediction from a "high-confidence wrong" flag, and Option B in Section 06 is built entirely
on a version of this number. Before relying on it, look at what actually happens to precision
and recall as the threshold moves.

In [ ]:
pr_data = plot_precision_recall_with_threshold(
    ham_results["probabilities"], ham_results["true_labels"], chosen_thresh=CONFIDENCE_THRESH
)

---
## 🔬 SECTION 04 — THE STRESS TEST
*Same model. Same code. A dataset built to find exactly this kind of failure.*

---

The MSKCC Skin Tone Labeling Dataset (ISIC Collection 413) was collected from 64 patients at
Memorial Sloan Kettering, recruited specifically to span all six Fitzpatrick skin types
evenly. Every lesion was examined by a board-certified dermatologist and confirmed benign at
the time of imaging — that's the whole point of the dataset; it exists to study skin-tone
labeling, not diagnosis. For this audit we use the **Fitzpatrick V-VI subset** specifically —
the skin tones most underrepresented in HAM10000 — since that's the population the bias
question is actually about.

That gives us an unusual but genuinely useful audit: **there is no "correct" diagnosis to
score against, but there is a correct answer to "is this dangerous" — and the answer is always
no.** Any time our model predicts melanoma, BCC, or actinic keratoses on one of these images,
that is, by construction, a false alarm.

In [ ]:
# Load ISIC 413 metadata and locate downloaded images
isic_metadata_path = next(ISIC_DIR.glob("*metadata*.csv"))
isic_df = pd.read_csv(isic_metadata_path)
print(f"✓ Loaded ISIC 413 metadata: {len(isic_df)} rows")
print(f"  Columns: {list(isic_df.columns)}")

# Find the Fitzpatrick column — ISIC metadata field names have varied across exports
fitz_col_candidates = [c for c in isic_df.columns if "fitzpatrick" in c.lower()]
print(f"  Fitzpatrick column detected: {fitz_col_candidates}")

In [ ]:
# Build image_id -> Fitzpatrick lookup and locate image files
FITZ_COL = fitz_col_candidates[0] if fitz_col_candidates else None

if FITZ_COL is None:
    print("⚠ No Fitzpatrick column found automatically — inspect isic_df.columns and set FITZ_COL manually.")
else:
    id_col = "isic_id" if "isic_id" in isic_df.columns else isic_df.columns[0]
    fitzpatrick_lookup = dict(zip(isic_df[id_col], isic_df[FITZ_COL]))
    # Fitzpatrick types may come through as roman numerals or "I"-"VI" strings — normalise to 1-6
    roman_map = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
    fitzpatrick_lookup = {
        k: (roman_map.get(str(v).strip(), v) if not str(v).strip().isdigit() else int(v))
        for k, v in fitzpatrick_lookup.items() if pd.notna(v)
    }
    print(f"✓ Fitzpatrick lookup built for {len(fitzpatrick_lookup)} images")
    print(f"  Distribution: {dict(sorted(Counter(fitzpatrick_lookup.values()).items()))}")

isic_image_paths = sorted(ISIC_DIR.glob("**/*.jpg")) + sorted(ISIC_DIR.glob("**/*.JPG"))
isic_image_paths = [p for p in isic_image_paths if p.stem in fitzpatrick_lookup]
print(f"✓ Matched {len(isic_image_paths)} images with known Fitzpatrick type")

In [ ]:
# Look at the training-vs-stress-test skin tone gap before running a single prediction
ham_fitz_estimated = {1: 35, 2: 35, 3: 20, 4: 7, 5: 2, 6: 1}  # estimated from HAM10000 collection geography (Austria/Australia)
afr_fitz_measured = dict(Counter(fitzpatrick_lookup.values()))

plot_fitzpatrick_distribution(ham_fitz_estimated, dataset_name="HAM10000 — Training Data (estimated)")
plot_fitzpatrick_distribution(afr_fitz_measured, dataset_name="ISIC 413 — Stress Test (Fitzpatrick V-VI only)")

In [ ]:
# Build the stress-test loader and run inference
stress_ds = StressTestDataset(isic_image_paths, fitzpatrick_lookup, EVAL_TRANSFORM)
stress_loader = DataLoader(stress_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
print(f"✓ Stress-test set: {len(stress_ds)} confirmed-benign images, Fitzpatrick V-VI")

stress_results = run_inference_unlabeled(model, stress_loader, DEVICE)
print("✓ Stress-test inference complete")

In [ ]:
# Build the stress-test table — every prediction of a malignant class is a false alarm,
# since every lesion here was confirmed benign
malignant_idx = [CLASS_NAMES.index(c) for c in MALIGNANT]

stress_rows = []
for img_id, pred, prob, fitz in zip(stress_results["image_ids"], stress_results["predictions"],
                                     stress_results["probabilities"], stress_results["fitzpatrick"]):
    conf = float(max(prob))
    stress_rows.append({
        "image_id": img_id,
        "fitzpatrick": fitz,
        "predicted_label": CLASS_LABELS[CLASS_NAMES[pred]],
        "predicted_malignant": pred in malignant_idx,
        "confidence": round(conf, 4),
    })
stress_table = pd.DataFrame(stress_rows)

overall_false_alarm_rate = stress_table["predicted_malignant"].mean()
print(f"Overall false-alarm rate on confirmed-benign dark-skin-tone images: {overall_false_alarm_rate:.1%}")
print(f"({stress_table['predicted_malignant'].sum()} of {len(stress_table)} confirmed-benign lesions flagged as malignant)")

In [ ]:
fitz_breakdown = plot_overdiagnosis_by_fitzpatrick(stress_table)
fitz_breakdown

In [ ]:
plot_confidence_by_fitzpatrick(stress_table)

---
#### ✍️ Reflection 2 — The Performance Gap

Fill in from what you've observed:

| Metric | HAM10000 (test) | Stress test (Fitzpatrick V-VI) |
|--------|----------|-----------------|
| Overall accuracy / false-alarm rate | | |
| Avg. confidence | | |

1. The model is similarly confident on both datasets. What does the false-alarm rate tell you about the model's calibration on skin tones it never trained on?
2. **You wrote down an accuracy threshold in Reflection 1 that would change your decision.** Does the false-alarm rate on Fitzpatrick V-VI cross that line?
3. A colleague argues the gap is just because phone-camera and dermoscope images differ in quality — not a skin-tone bias issue. ISIC 413 images are dermoscopic, like the training data, not phone photos. How does that change your colleague's argument?
4. Of the high-confidence false alarms (confidence ≥ 0.80) on confirmed-benign lesions, which class is the model most often mistaking benign skin for? What might that mean clinically — unnecessary biopsies, patient anxiety, lost trust in the tool?

---

*Your answer here.*

---

In [ ]:
# Worst false alarms — highest-confidence malignant predictions on confirmed-benign images,
# grouped by error type and by Fitzpatrick type for the visual error analysis
false_alarms = stress_table[stress_table["predicted_malignant"]].sort_values("confidence", ascending=False)

print(f"Total false alarms: {len(false_alarms)}")
print(f"\nFalse alarms by predicted class:")
print(false_alarms["predicted_label"].value_counts())
print(f"\nFalse alarms by Fitzpatrick type:")
print(false_alarms.groupby("fitzpatrick").size())

In [ ]:
# Display the 8 highest-confidence false alarms — the model at its most dangerously wrong
worst_false_alarms = false_alarms.head(8)
id_to_path = {p.stem: p for p in isic_image_paths}

show_imgs, show_preds, show_confs, show_fitz = [], [], [], []
for _, row in worst_false_alarms.iterrows():
    path = id_to_path.get(row["image_id"])
    if path:
        idx = [i for i, p in enumerate(isic_image_paths) if p.stem == row["image_id"]][0]
        tensor, _, fitz = stress_ds[idx]
        show_imgs.append(tensor)
        show_preds.append(row["predicted_label"])
        show_confs.append(row["confidence"])
        show_fitz.append(int(row["fitzpatrick"]))

if show_imgs:
    display_stress_grid(show_imgs, show_preds, show_confs, show_fitz, n=len(show_imgs),
                        title="Highest-Confidence False Alarms — Confirmed-Benign, Dark Skin Tones")
else:
    print("⚠ Could not locate image files for display — check isic_image_paths.")

In [ ]:
# The comparison that actually matters: how the model behaves on its own home turf
# (HAM10000, mostly light skin tones) vs. on confirmed-benign Fitzpatrick V-VI images.
# This stress-test set is Fitzpatrick V-VI only by design, so there's no light-skin
# subgroup to compare against within it — the real comparison is across datasets.
ham_overall_accuracy = ham_metrics["overall_accuracy"]
isic_false_alarm_rate = stress_table["predicted_malignant"].mean()

print(f"HAM10000 held-out test — overall accuracy:        {ham_overall_accuracy:.1%}")
print(f"ISIC 413 (Fitzpatrick V-VI) — false-alarm rate:    {isic_false_alarm_rate:.1%}")
print(f"  (every one of these {len(stress_table)} lesions was confirmed benign by a dermatologist)")

fig, ax = plt.subplots(figsize=(6,4))
bars = ax.bar(
    ["HAM10000\n(mostly light skin)\nerror rate", "ISIC 413\n(Fitzpatrick V-VI)\nfalse-alarm rate"],
    [(1 - ham_overall_accuracy) * 100, isic_false_alarm_rate * 100],
    color=[TEAL, AMBER], edgecolor="white",
)
for bar, r in zip(bars, [(1 - ham_overall_accuracy) * 100, isic_false_alarm_rate * 100]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{r:.1f}%",
            ha="center", fontsize=10, color=NAVY)
ax.set_ylabel("Rate (%)", color=NAVY)
ax.set_title("Training-Set Error Rate vs Stress-Test False-Alarm Rate",
            fontsize=12, fontweight="bold", color=NAVY)
plt.tight_layout(); plt.show()

---
#### 💡 Optional extension — Grad-CAM on a false alarm

If you want to go one level deeper: load a Grad-CAM library (e.g. `pip install grad-cam`)
and visualise *where* in a false-alarm image the model is looking. Does it focus on the lesion,
or on skin pigmentation around it? This is genuinely useful for root-causing whether the model
learned "darker background pixels correlate with malignant labels" during training — a known
failure mode in dermatology AI when training data lacks darker skin tones. We leave this as an
extension rather than a required step, since it adds a new dependency and a few extra minutes
of setup; ask your facilitator if you want to try it live.

---

---
## 🔍 SECTION 05 — ROOT CAUSE
*The failure is documented. Now trace it. Where did it come from?*

---

Read the HAM10000 datasheet below. Pay attention to collection geography, imaging equipment,
and what demographic data was — and wasn't — collected.

In [ ]:
# Display the HAM10000 datasheet from the cloned repo
from IPython.display import Markdown, display

DATASHEET = Path("IndabaX-2026/Ethical Computer Vision/data/ham10000_datasheet.md")
if DATASHEET.exists():
    display(Markdown(DATASHEET.read_text()))
else:
    print("⚠ Datasheet not found at:", DATASHEET)
    print("\nSummary of key facts about HAM10000:")
    print("  - Collected in Austria (ViDIR group, Medical University of Vienna) and Australia (QIMR Berghofer)")
    print("  - Images acquired with dermatoscopes — not phone cameras")
    print("  - Estimated Fitzpatrick distribution: ~70% Type I-II, ~25% Type III, ~5% Type IV-VI")
    print("  - No systematic per-patient demographic metadata collected")
    print("  - No African sites included in collection")

---
#### ✍️ Reflection 3 — Tracing the Failure

1. The datasheet lists collection sites. Which countries are represented? Which continent is absent?
2. The model was fine-tuned on HAM10000. Explain — in plain language, without jargon — why a model trained on this data would generate more false alarms on Fitzpatrick V-VI skin, even though it was never tested against an actual melanoma case in that group.
3. Is this a problem with the model architecture (MobileNetV3-Small vs. a bigger model)? With the training code? Or with something more fundamental? What would you change?
4. A lawyer argues the startup is liable if the model causes unnecessary biopsies on patients with darker skin. A technologist argues the model is working as designed — it just wasn't designed for this population. Who is right?

---

*Your answer here.*

---

---
## 🛤️ SECTION 06 — PROPOSE A PATH
*Analysis is not enough. An auditor's job is to make a recommendation.*

---

There is no single right answer. Three credible paths exist, each with real trade-offs.

> ⚖️ **Three paths forward:**

**Option A — Collect and fine-tune**
Gather 500-1,000 dermatoscopic or phone-camera images from African clinical settings with
proper consent and Fitzpatrick labels, paired with confirmed diagnoses (not just confirmed-benign
controls). Fine-tune the model on this data. Re-audit before deployment. Estimated timeline:
6-12 months.

*Trade-off:* takes time and resources; the right thing to do for long-term safety. Requires
community partnerships and ethical data collection protocols — and critically, requires
*diagnosed* cases, not just skin-tone reference images, since ISIC 413 alone cannot validate
diagnostic accuracy.

**Option B — Restricted deployment with mandatory human review**
Deploy immediately but restrict the model to a decision-support role only. Every high-confidence
prediction triggers mandatory dermatologist review before any clinical action. The model cannot
act autonomously.

*Trade-off:* allows faster deployment; reduces but does not eliminate harm; adds operational
cost; the false-alarm-rate gap you measured means dark-skinned patients will be referred for
unnecessary review more often than light-skinned patients — a real, if lesser, harm.

**Option C — Halt and rebuild**
Do not deploy. Return to the startup with a documented audit report. Recommend starting from a
more representative, *diagnosed* dataset (Fitzpatrick17k + SCIN, both with malignant/benign
labels across skin tones) before returning to deployment discussion.

*Trade-off:* the safest option for patients; significant commercial cost to the startup; delays
access to any tool for community health workers.

In [ ]:
def plot_options_comparison(figsize=(8,4)):
    _style()
    options = ["Option A\nCollect & Fine-tune", "Option B\nRestricted + Human Review", "Option C\nHalt Deployment"]
    dimensions = ["Time to\nDeployment", "Data Collection\nBurden", "Risk to\nPatient", "Risk to\nStartup"]
    values = np.array([[2,2,1,1],[1,0,1,1],[0,0,0,2]])
    labels = np.array([["High","High","Med","Low"], ["Med","Low","Med","Med"], ["—","—","Low","High"]])
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(values, cmap=plt.cm.RdYlGn_r, norm=plt.Normalize(0,2), aspect="auto")
    ax.set_xticks(range(4)); ax.set_xticklabels(dimensions, fontsize=9, color=NAVY)
    ax.set_yticks(range(3)); ax.set_yticklabels(options, fontsize=9, color=NAVY)
    ax.xaxis.set_ticks_position("top"); ax.xaxis.set_label_position("top")
    for i in range(3):
        for j in range(4):
            ax.text(j, i, labels[i,j], ha="center", va="center", fontsize=11,
                    fontweight="bold", color="white" if values[i,j]==2 else NAVY)
    ax.set_title("Deployment Options — Risk and Burden Comparison", fontsize=12, fontweight="bold", color=NAVY, pad=30)
    handles = [mpatches.Patch(color=c, label=l) for c, l in [("#4CAF50","Low"),("#FFC107","Med"),("#F44336","High")]]
    ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(1.0,-0.18), ncol=3, fontsize=8, frameon=True)
    plt.tight_layout(); plt.show()

plot_options_comparison()

---

**Real-world precedents:**

**Option A — The SCIN template.** In 2024, Google Research released the Skin Condition Image
Network (SCIN) — over 10,000 diverse skin images collected with explicit Fitzpatrick labels
from a global community of contributors, paired with clinician-reviewed diagnoses. This is what
responsible, *diagnosed* collection at scale looks like. [Read more →](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/)

**Option B — The FDA framework.** The US FDA's guidance on AI/ML-based Software as a Medical
Device (SaMD) requires pre-specified performance goals on demographically representative test
sets. A human-in-the-loop requirement is one path to approval for tools that do not yet meet
those goals.

**Option C — The pulse oximeter precedent.** Pulse oximeters were deployed for decades before
Sjoding et al. (NEJM 2020) found they systematically over-read on darker skin — leading to
COVID-19 patients being denied oxygen they needed. The cost of that failure vastly exceeded the
cost of earlier detection. [Read the paper →](https://www.nejm.org/doi/full/10.1056/NEJMc2029240)

---

#### 🗳️ Quick poll — before you write your recommendation

*Which option do you lean toward right now, before formally writing it up? Note it, then see if
your formal answer below matches your gut reaction.*

---

---
#### ✍️ Reflection 4 — Your Recommendation

Select your recommended path and defend it in 3-5 sentences.

**Option A / B / C** *(delete two)*

**Recommendation:**
*Write here.*

**Justification:**
*Write here — reference the specific false-alarm rate and confidence numbers you observed on Fitzpatrick V-VI.*

**Conditions:**
*What conditions must be met before any deployment? What false-alarm-rate parity across Fitzpatrick types would you require the startup to demonstrate?*

---

*Your answer here.*

---

## Conclusion

The startup's CTO is waiting for your answer. The health workers in Brong-Ahafo are waiting for
the app. These are not abstract trade-offs — they are decisions that will affect real patients.

What you did in this audit: you didn't just inherit someone else's model and run it through a
checklist. You trained it — chose the backbone, watched the loss curve, picked the threshold —
and then you watched it fail in a specific, measurable, skin-tone-correlated way on data it had
never seen. The gap is real. The false-alarm pattern is real. The fact that a fairness-research
dataset like ISIC 413 doesn't even carry diagnostic labels is itself a finding: the tools to
audit dermatology AI on African and dark-skinned patients properly barely exist yet.

The technical skills you used — fine-tuning a backbone, computing metrics by hand, tracing a
threshold to a precision/recall trade-off, grouping failures by demographic attribute — are the
same skills used in actual ML safety work. The difference between a responsible deployment and
a harmful one is often just whether someone ran these cells and acted on what they found.

---

📚 **Key references from this audit:**

| Resource | Why |
|---|---|
| [HAM10000 — Harvard Dataverse](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T) | The training dataset — read the metadata |
| [MSKCC Skin Tone Labeling Dataset (ISIC Collection 413)](https://api.isic-archive.com/collections/413/) | The stress-test set used in this audit |
| [Fitzpatrick17k](https://github.com/mattgroh/fitzpatrick17k) | A diagnostic dataset with skin-tone labels — what a true fine-tuning set should look like |
| [SCIN Dataset — Google 2024](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/) | How to collect representative, diagnosed skin data responsibly |
| [Racial Bias in Pulse Oximetry — NEJM 2020](https://www.nejm.org/doi/full/10.1056/NEJMc2029240) | The cost of deploying a biased medical device |
| [Model Cards — Mitchell et al. 2018](https://arxiv.org/abs/1810.03993) | What documentation should accompany every model |
| [Datasheets for Datasets — Gebru et al. 2018](https://arxiv.org/abs/1803.09010) | What documentation should accompany every dataset |
| [FDA AI/ML SaMD Guidance](https://www.fda.gov/medical-devices/software-medical-device-samd/artificial-intelligence-and-machine-learning-software-medical-device) | The regulatory framework for medical AI |
| [Searching for a Black Cat in a Dark Room (talk on dataset bias)](https://www.youtube.com/results?search_query=dataset+bias+machine+learning+talk) | Search for current talks on dataset bias — several good ones circulate each year |

---

*Part 1 of this tutorial covers the computer vision foundations behind this model: how images
become tensors, how preprocessing pipelines work, how CNNs learn, and how YOLO, Vision
Transformers, and CLIP extend the toolkit.*